# 01. Разведочный анализ данных

Цель: понять структуру сетки, проверить физические инварианты, найти нулевую поверхность VV-каналов.

Результат этого ноутбука — не модель, а список фактов о данных, на которых строится всё остальное.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

from co2relax.data import (
    load, grid_summary, check_invariants, vt2_t3_spread, sign_crossings,
    FIG_DIR, TEMP_COLS, R_COLS, TARGETS, THETA_2, THETA_3,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4g}".format)
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = load()
print(df.shape)
df.head()

## 1. Структура сетки

Какие значения принимают температуры, сколько партнёров и плотностей, полна ли сетка.

In [ ]:
g = grid_summary(df)
for c in ["T", "T12", "T3"]:
    v = g[c]
    print(f"{c:4s}: {len(v)} значений, {v[0]} … {v[-1]}, шаги {sorted(set(np.diff(v).tolist()))}")
print("partner:", g["partner"])
print("n_CO2:  ", g["n_CO2"])
print()
pprint({k: g[k] for k in ["rows", "full_grid_rows", "grid_is_full", "duplicates_by_key", "nan_total"]})

## 2. Физические инварианты

1. `R_VT2_3 ≡ 0` — VT2 не трогает моду 3
2. `R_*_3 = −R_*_12 / 3` — размен 1 квант ν₃ на 3 кванта ν₂
3. `R = 0` при `T = T12 = T3` — детальный баланс
4. `R ∝ n²` — билинейность по заселённостям

In [ ]:
inv = check_invariants(df)
pprint(inv)

## 3. Диапазоны величин

В логарифме — иначе числа порядка 1e29 нечитаемы. Плюс доля отрицательных и нулевых значений.

In [ ]:
logR = df[TARGETS].abs().replace(0, np.nan).apply(np.log10)
display(logR.describe())

summary = pd.DataFrame({
    "доля < 0": (df[TARGETS] < 0).mean(),
    "доля = 0": (df[TARGETS] == 0).mean(),
    "доля > 0": (df[TARGETS] > 0).mean(),
})
summary

## 4. Зависит ли VT2 от T3

Относительный размах `R_VT2_12` по `T3` при фиксированных `(T, T12)`.
Если везде ~1e-6 — VT2 фактически функция двух переменных.

In [ ]:
spread = vt2_t3_spread(df)
spread.describe()

In [ ]:
spread = vt2_t3_spread(df)
s = spread.reset_index()
print(s.nlargest(10, "R_VT2_12"))

In [ ]:
keys = ["partner", "T", "T12", "T3"]
print("строк:", len(df))
print("уникальных точек:", df[keys].drop_duplicates().shape[0])
for c in ["T", "T12", "T3"]:
    v = sorted(df[c].unique())
    print(f"{c}: {len(v)} значений, {v[0]}…{v[-1]}, шаг {sorted(set(np.diff(v).tolist()))}")

In [ ]:
core = df.drop_duplicates(subset=keys).copy()
for c in TARGETS:
    core[c] = core[c] / core["n_CO2"]**2
core = core.drop(columns=["n_CO2"]).reset_index(drop=True)
print(core.shape)   # (729, 7)

In [ ]:
print((core["T12"] < core["T"]).sum(), (core["R_VT2_12"] > 0).sum())

## 5. Срезы: как ведёт себя R_VV23_12 по T3


In [ ]:
T_fix = 1000
sub = core[core["T"] == T_fix]

fig, ax = plt.subplots(figsize=(7, 4.5))
for t12, gg in sub.groupby("T12"):
    gg = gg.sort_values("T3")
    ax.plot(gg["T3"], np.arcsinh(gg["R_VV23_12"] / 1e-19), "o-", ms=4, label=f"T12={t12:g}")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("T3, K"); ax.set_ylabel("asinh(R / 1e-19)")
ax.set_title(f"T = {T_fix} K"); ax.legend(fontsize=7, ncol=2)

In [ ]:
zc = sign_crossings(core, "R_VV23_12")
print(len(zc))
zc.pivot_table(index="T12", columns="T", values="T3_zero").round(0)

`asinh` вместо `log` — чтобы видеть и положительные, и отрицательные значения на одной оси.

## 6. Карта знака

Для нескольких значений `T`: красное — мода 12 получает кванты, синее — отдаёт.
Граница между цветами — нулевая поверхность.

In [ ]:
T_vals = sorted(df["T"].unique())
T_show = T_vals[:: max(1, len(T_vals) // 4)][:4]

fig, axes = plt.subplots(1, len(T_show), figsize=(4 * len(T_show), 3.8), sharey=True)
axes = np.atleast_1d(axes)
for ax, T in zip(axes, T_show):
    piv = df[df["T"] == T].pivot_table(index="T12", columns="T3", values="R_VV23_12")
    ax.pcolormesh(piv.columns, piv.index, np.sign(piv.values),
                  cmap="coolwarm", vmin=-1, vmax=1, shading="nearest")
    ax.plot(piv.columns, piv.columns, "k--", lw=0.6)
    ax.set_title(f"T = {T:g} K"); ax.set_xlabel("T3, K")
axes[0].set_ylabel("T12, K")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_vv23_sign_map.png", dpi=150, bbox_inches="tight")

## 7. Нулевая поверхность против резонансного условия

Наивное резонансное условие без учёта дефекта энергии: `3θ₂/T12 = θ₃/T3`,
то есть `T3 = (θ₃ / 3θ₂) · T12 ≈ 1.17 · T12`.

Если реальная граница систематически отличается — это вклад дефекта энергии и газовой температуры `T`.
Именно это нужно понять, чтобы построить сродство `φ` для факторизации таргета.

In [ ]:
# наивный подход без учета дефекта энергии (зависимости от T)
zc = sign_crossings(df, "R_VV23_12")
print(f"найдено пересечений нуля: {len(zc)}")

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(zc["T12"], zc["T3_zero"], c=zc["T"], cmap="viridis", s=18)
t = np.linspace(df["T12"].min(), df["T12"].max(), 50)
ax.plot(t, THETA_3 / (3 * THETA_2) * t, "r--", label="наивный резонанс")
ax.plot(t, t, "k:", lw=0.8, label="T3 = T12")
ax.plot(t, THETA_3 / (3 * THETA_2) * t, "r--",
        label=r"$3\theta_2/T_{12} = \theta_3/T_3$")
ax.set_xlabel("T12, K"); ax.set_ylabel("T3 на нуле, K")
fig.colorbar(sc, label="T, K"); ax.legend()
fig.savefig(FIG_DIR / "01_zero_surface.png", dpi=150, bbox_inches="tight")

zc.head(10)

In [ ]:
D = THETA_3 - 3*THETA_2          # 500 K

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(zc["T12"], zc["T3_zero"], c=zc["T"], cmap="viridis", s=22, zorder=3)

t12 = np.linspace(1000, 5000, 200)
for T in [1000, 2000, 3000, 5000]:
    t3 = THETA_3 / (3*THETA_2/t12 + D/T)
    ax.plot(t12, t3, "--", lw=1, alpha=0.7, label=f"теория, T={T}")

ax.set_xlabel("T12, K"); ax.set_ylabel("T3 на нуле, K")
fig.colorbar(sc, label="T, K"); ax.legend(fontsize=8)
fig.savefig(FIG_DIR / "01_zero_surface.png", dpi=150, bbox_inches="tight")

In [ ]:
# Зависит ли положение нуля от T? Если да — дефект энергии влияет заметно.
zc.pivot_table(index="T12", columns="T", values="T3_zero").round(0)

In [ ]:
zc123 = sign_crossings(core, "R_VV123_12")
t123 = zc123.pivot(index="T12", columns="T", values="T3_zero")
t23  = sign_crossings(core, "R_VV23_12").pivot(index="T12", columns="T", values="T3_zero")
d = (t123 - t23).abs()
print(f"медиана {d.stack().median():.1f} K, max {d.max().max():.1f} K")

In [ ]:
d = (t123 - t23).abs()
print(d.stack().idxmax(), d.max().max())

## Выводы


**Структура данных.** Файл содержит 3 645 000 строк, из которых физически
различных точек 729. Сетка по температурам полная и регулярная:
T, T12, T3 ∈ {1000, 1500, …, 5000} K, шаг 500 K. Единственный партнёр — CO₂.
Плотность n_CO2 пробегает ~5000 значений от 1·10²³ до 1·10²⁵ м⁻³.

**Инварианты.** Все четыре физических соотношения подтверждены:

| Соотношение | Результат |
|---|---|
| R_VT2_3 ≡ 0 | выполняется точно |
| R_*_3 = −R_*_12/3 для VV23 и VV123 | 1·10⁻¹⁴ |
| R = 0 при T = T12 = T3 | ровно 0.0 (45 000 строк) |
| R ∝ n² | 2.7·10⁻¹⁴ |

Последнее позволяет исключить плотность из числа аргументов: после деления
таргетов на n² величина от неё не зависит и приобретает смысл эффективного
коэффициента скорости (м³/с). Размерность задачи сокращается до трёх входов.

**Зависимость VT2 от T3.** Гипотеза о независимости не подтвердилась, однако
эффект мал: относительный размах R_VT2_12 при фиксированных (T, T12) не
превышает 2·10⁻³, медиана 3·10⁻⁵. Зависимость косвенная — процесс VT2 не
изменяет i₃, но T3 входит в статистическую сумму квазистационарного
распределения. Максимум размаха достигается при минимальной T и максимальной
T12, что согласуется с указанным механизмом. Все три канала моделируются как
функции трёх температур.

**Область определения.** Обе ветви VT-процесса представлены: 324 точки с
T12 < T (накачка, R_VT2_12 > 0), 324 точки с T12 > T (разгрузка,
R_VT2_12 < 0), 81 точка на плоскости T12 = T (R = 0). Соответствие знака
и соотношения температур выполняется без исключений.

**Структура VV-каналов.** Релаксационные члены VV23 и VV123 меняют знак:
при низких T3 мода ν₃ принимает кванты, при высоких — отдаёт. В логарифмическом
масштабе профили имеют характерный вид с двумя плато и резким переходом,
что отвечает представлению R = A·[1 − exp(−φ)].

**Нулевая поверхность.** Положение границы смены знака извлечено линейной
интерполяцией между узлами сетки и сопоставлено с двумя моделями.
Резонансное условие 3θ₂/T₁₂ = θ₃/T₃, не учитывающее энергетический дефект,
задаёт единственную прямую, систематически завышает T₃ и не описывает
зависимость от T. Условие обращения в ноль сродства процесса

    φ = θ₃/T₃ − 3θ₂/T₁₂ − Δ/T,    Δ = θ₃ − 3θ₂ = 500 K

со спектроскопическими значениями θ₂ = 960 K, θ₃ = 3380 K и без подгоняемых
параметров воспроизводит данные со среднеквадратичным отклонением 11 K
(2.2% шага сетки), медианная относительная ошибка 0.34%. Контрольная подгонка
Δ как свободного параметра даёт 502.4 K.

Тождество Δ = θ₃ − 3θ₂ обеспечивает φ ≡ 0 при T = T12 = T3, то есть детальный
баланс выполняется структурно, а не за счёт подгонки.

**Совпадение каналов.** Нулевые поверхности VV23 и VV123 совпадают: медианное
расхождение 5 K, максимальное 36 K при шаге сетки 500 K, что не превышает
погрешности интерполяции нуля (11 K). Максимум достигается при T12 = 5000,
T = 1000 — в области наибольшей кривизны профиля, где линейная интерполяция
наименее точна. Совпадение отвечает практическому равенству энергетических
дефектов: θ₃ − 3θ₂ = 500 K и θ₃ − θ₁ − θ₂ = 502 K вследствие резонанса Ферми
θ₁ ≈ 2θ₂. Оба канала описываются единым сродством φ и различаются только
предэкспоненциальным множителем A.

**Следствия для модели.**

1. Входы: (T, T12, T3). Выходы: R_VT2_12, R_VV23_12, R_VV123_12; остальные
   три столбца восстанавливаются точно.
2. Плотность выносится аналитически множителем n².
3. Таргет представляется в виде R = A(T, T12, T3)·[1 − exp(−φ)], где φ
   вычисляется по спектроскопическим константам. Обучению подлежит только
   гладкий множитель A, что гарантирует точное соблюдение знака, положения
   нулевой поверхности и равенства нулю на равновесии.
4. Объём выборки (729 точек) требует валидации методом leave-one-out или
   k-fold по срезам; отдельная тестовая выборка нерепрезентативна.

**Объём задачи.** После исключения плотности датасет содержит 729 точек при
трёх входных переменных. При таком объёме физически мотивированная
параметризация таргета предпочтительнее свободной аппроксимации: структура,
известная аналитически, задаётся явно, а обучению подлежит только гладкий
остаток.